##### Getting data (domestic and international tourist demand) from INE API. Cleaning, preparing, and saving final dataset as a csv file

In [3]:
import requests
import pandas as pd

In [4]:
url = "https://servicios.ine.es/wstempus/js/EN/DATOS_TABLA/2078?date=20210101:20261231"

response = requests.get(url)
response.status_code

200

In [5]:
# getting data from api

data = response.json()
type(data), len(data)

(list, 552)

In [6]:
# checking data

data[0]

{'COD': 'EOT2611',
 'Nombre': 'National. Travellers. 01059-Vitoria-Gastéiz. Residents in Spain.',
 'FK_Unidad': 213,
 'FK_Escala': 1,
 'Data': [{'Fecha': 1785535200000,
   'FK_TipoDato': 2,
   'FK_Periodo': 8,
   'Anyo': 2026,
   'Valor': 18358.0,
   'Secreto': False},
  {'Fecha': 1782856800000,
   'FK_TipoDato': 2,
   'FK_Periodo': 7,
   'Anyo': 2026,
   'Valor': 18734.0,
   'Secreto': False},
  {'Fecha': 1780264800000,
   'FK_TipoDato': 2,
   'FK_Periodo': 6,
   'Anyo': 2026,
   'Valor': 19464.0,
   'Secreto': False},
  {'Fecha': 1777586400000,
   'FK_TipoDato': 2,
   'FK_Periodo': 5,
   'Anyo': 2026,
   'Valor': 20574.0,
   'Secreto': False},
  {'Fecha': 1774994400000,
   'FK_TipoDato': 2,
   'FK_Periodo': 4,
   'Anyo': 2026,
   'Valor': 21012.0,
   'Secreto': False},
  {'Fecha': 1772319600000,
   'FK_TipoDato': 2,
   'FK_Periodo': 3,
   'Anyo': 2026,
   'Valor': 20189.0,
   'Secreto': False},
  {'Fecha': 1769900400000,
   'FK_TipoDato': 2,
   'FK_Periodo': 2,
   'Anyo': 2026,
   'V

In [7]:
# filtering Barcelona only

barcelona_data = []

for item in data:
    if "Barcelona" in item["Nombre"]:
        barcelona_data.append(item)

len(barcelona_data)        

4

In [8]:
# checking series description

for item in barcelona_data:
    print(item["Nombre"])

National. Travellers. 08019-Barcelona. Residents in Spain.
National. Travellers. 08019-Barcelona. Residents abroad.
National. Overnight stays. 08019-Barcelona. Residents in Spain.
National. Overnight stays. 08019-Barcelona. Residents abroad.


In [9]:
# every month turning into 1 row
rows = []

for item in barcelona_data:
    for record in item["Data"]:
        rows.append({
            "series": item["Nombre"],
            "year": record["Anyo"],
            "month": record["FK_Periodo"],
            "value": record["Valor"]
        })

# getting all the rows into a table
df_barcelona_ine = pd.DataFrame(rows)

df_barcelona_ine.head()

,series,year,month,value
0,National. Travellers. 08019-Barcelona. Residen...,2026,8,100991.0
1,National. Travellers. 08019-Barcelona. Residen...,2026,7,114427.0
2,National. Travellers. 08019-Barcelona. Residen...,2026,6,133381.0
3,National. Travellers. 08019-Barcelona. Residen...,2026,5,137723.0
4,National. Travellers. 08019-Barcelona. Residen...,2026,4,140825.0


In [12]:
# changing names of the future columns

df_barcelona_ine["series"] = df_barcelona_ine["series"].replace({
    "National. Travellers. 08019-Barcelona. Residents in Spain.": "travellers_spain",
    "National. Travellers. 08019-Barcelona. Residents abroad.": "travellers_abroad",
    "National. Overnight stays. 08019-Barcelona. Residents in Spain.": "nights_spain",
    "National. Overnight stays. 08019-Barcelona. Residents abroad.": "nights_abroad"
})

df_barcelona_ine["series"].unique()

<ArrowStringArray>
['travellers_spain', 'travellers_abroad', 'nights_spain', 'nights_abroad']
Length: 4, dtype: str

In [14]:
#turning table (1 month = 1 row, columns = travellers_spain, travellers_abroad, nights_spain, nights_abroad )

df_barcelona_ine_1 = df_barcelona_ine.pivot(
    index=["year", "month"],
    columns="series",
    values="value"
).reset_index()

df_barcelona_ine_1

series,year,month,nights_abroad,nights_spain,travellers_abroad,travellers_spain
0,2021,1,49018.0,75385.0,22102.0,42324.0
1,2021,2,51982.0,89137.0,22724.0,55121.0
2,2021,3,74868.0,106209.0,32200.0,64146.0
3,2021,4,103099.0,107864.0,40560.0,59229.0
4,2021,5,223626.0,137286.0,87339.0,75615.0
...,...,...,...,...,...,...
63,2026,4,1660919.0,264703.0,643964.0,140825.0
64,2026,5,1748244.0,247095.0,717118.0,137723.0
65,2026,6,1678024.0,235296.0,703381.0,133381.0
66,2026,7,1889694.0,209672.0,762340.0,114427.0


In [15]:
# making total nights and total travellers

df_barcelona_ine_1["total_travellers"] = (
    df_barcelona_ine_1["travellers_abroad"] +
    df_barcelona_ine_1["travellers_spain"]
)

df_barcelona_ine_1["total_nights"] = (
    df_barcelona_ine_1["nights_abroad"] +
    df_barcelona_ine_1["nights_spain"]
)

df_barcelona_ine_1.head()

series,year,month,nights_abroad,nights_spain,travellers_abroad,travellers_spain,total_travellers,total_nights
0,2021,1,49018.0,75385.0,22102.0,42324.0,64426.0,124403.0
1,2021,2,51982.0,89137.0,22724.0,55121.0,77845.0,141119.0
2,2021,3,74868.0,106209.0,32200.0,64146.0,96346.0,181077.0
3,2021,4,103099.0,107864.0,40560.0,59229.0,99789.0,210963.0
4,2021,5,223626.0,137286.0,87339.0,75615.0,162954.0,360912.0


In [ ]:
# making proper "date" column

df_barcelona_ine_1["date"] = pd.to_datetime(
    dict(
        year=df_barcelona_ine_1["year"],
        month=df_barcelona_ine_1["month"],
        day=1
    )
)

df_barcelona_ine_1

series,year,month,nights_abroad,nights_spain,travellers_abroad,travellers_spain,total_travellers,total_nights,date
0,2021,1,49018.0,75385.0,22102.0,42324.0,64426.0,124403.0,2021-01-01
1,2021,2,51982.0,89137.0,22724.0,55121.0,77845.0,141119.0,2021-02-01
2,2021,3,74868.0,106209.0,32200.0,64146.0,96346.0,181077.0,2021-03-01
3,2021,4,103099.0,107864.0,40560.0,59229.0,99789.0,210963.0,2021-04-01
4,2021,5,223626.0,137286.0,87339.0,75615.0,162954.0,360912.0,2021-05-01
...,...,...,...,...,...,...,...,...,...
63,2026,4,1660919.0,264703.0,643964.0,140825.0,784789.0,1925622.0,2026-04-01
64,2026,5,1748244.0,247095.0,717118.0,137723.0,854841.0,1995339.0,2026-05-01
65,2026,6,1678024.0,235296.0,703381.0,133381.0,836762.0,1913320.0,2026-06-01
66,2026,7,1889694.0,209672.0,762340.0,114427.0,876767.0,2099366.0,2026-07-01


In [17]:
# sorting by date

df_barcelona_ine_1 = df_barcelona_ine_1.sort_values("date").reset_index(drop=True)

In [18]:
# removing "series" (index column)

df_barcelona_ine_1.columns.name = None

In [ ]:
# checking

df_barcelona_ine_1

,year,month,nights_abroad,nights_spain,travellers_abroad,travellers_spain,total_travellers,total_nights,date
0,2021,1,49018.0,75385.0,22102.0,42324.0,64426.0,124403.0,2021-01-01
1,2021,2,51982.0,89137.0,22724.0,55121.0,77845.0,141119.0,2021-02-01
2,2021,3,74868.0,106209.0,32200.0,64146.0,96346.0,181077.0,2021-03-01
3,2021,4,103099.0,107864.0,40560.0,59229.0,99789.0,210963.0,2021-04-01
4,2021,5,223626.0,137286.0,87339.0,75615.0,162954.0,360912.0,2021-05-01
...,...,...,...,...,...,...,...,...,...
63,2026,4,1660919.0,264703.0,643964.0,140825.0,784789.0,1925622.0,2026-04-01
64,2026,5,1748244.0,247095.0,717118.0,137723.0,854841.0,1995339.0,2026-05-01
65,2026,6,1678024.0,235296.0,703381.0,133381.0,836762.0,1913320.0,2026-06-01
66,2026,7,1889694.0,209672.0,762340.0,114427.0,876767.0,2099366.0,2026-07-01


In [20]:
# checking if there are any null values

df_barcelona_ine_1.isna().sum()

year                 0
month                0
nights_abroad        0
nights_spain         0
travellers_abroad    0
travellers_spain     0
total_travellers     0
total_nights         0
date                 0
dtype: int64

In [21]:
# dropping year and month columns, as we now have date

df_barcelona_ine_1 = df_barcelona_ine_1.drop(columns=["year", "month"])

In [ ]:
# checking

df_barcelona_ine_1

,nights_abroad,nights_spain,travellers_abroad,travellers_spain,total_travellers,total_nights,date
0,49018.0,75385.0,22102.0,42324.0,64426.0,124403.0,2021-01-01
1,51982.0,89137.0,22724.0,55121.0,77845.0,141119.0,2021-02-01
2,74868.0,106209.0,32200.0,64146.0,96346.0,181077.0,2021-03-01
3,103099.0,107864.0,40560.0,59229.0,99789.0,210963.0,2021-04-01
4,223626.0,137286.0,87339.0,75615.0,162954.0,360912.0,2021-05-01
...,...,...,...,...,...,...,...
63,1660919.0,264703.0,643964.0,140825.0,784789.0,1925622.0,2026-04-01
64,1748244.0,247095.0,717118.0,137723.0,854841.0,1995339.0,2026-05-01
65,1678024.0,235296.0,703381.0,133381.0,836762.0,1913320.0,2026-06-01
66,1889694.0,209672.0,762340.0,114427.0,876767.0,2099366.0,2026-07-01


In [23]:
# rearenging logical columns order

df_barcelona_ine_final = df_barcelona_ine_1[
    [
        "date",
        "total_travellers",
        "travellers_spain",
        "travellers_abroad",
        "total_nights",
        "nights_spain",
        "nights_abroad"
    ]
]

In [ ]:
# clean final table

df_barcelona_ine_final.head()

,date,total_travellers,travellers_spain,travellers_abroad,total_nights,nights_spain,nights_abroad
0,2021-01-01,64426.0,42324.0,22102.0,124403.0,75385.0,49018.0
1,2021-02-01,77845.0,55121.0,22724.0,141119.0,89137.0,51982.0
2,2021-03-01,96346.0,64146.0,32200.0,181077.0,106209.0,74868.0
3,2021-04-01,99789.0,59229.0,40560.0,210963.0,107864.0,103099.0
4,2021-05-01,162954.0,75615.0,87339.0,360912.0,137286.0,223626.0


In [25]:
df_barcelona_ine_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 68 entries, 0 to 67
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   date               68 non-null     datetime64[us]
 1   total_travellers   68 non-null     float64       
 2   travellers_spain   68 non-null     float64       
 3   travellers_abroad  68 non-null     float64       
 4   total_nights       68 non-null     float64       
 5   nights_spain       68 non-null     float64       
 6   nights_abroad      68 non-null     float64       
dtypes: datetime64[us](1), float64(6)
memory usage: 3.8 KB


In [26]:
# saving to csv file

# df_barcelona_ine_final.to_csv("ine_barcelona.csv", index=False)